In [1]:
# 사전 설치 : pip install konlpy (konlpy의 Okt는 Java가 설치되어 있어야 작동)
# pip install JPype1==1.4.1   // Jpype1 : 파이썬 코드 내에서 Java 코드 호출 (cf. konlpy okt가 Java 엔진 사용 )
# jdk17 사전설치필요(환경변수 설정 포함)
# 실습은 한글경로(X), 반드시 영문폴더 경로 지정
from konlpy.tag import Okt    # 한국어 자연어처리 라이브러리 (형태소 분석)
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer 
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding
from tensorflow.keras.models import Sequential

In [2]:
# 1. 텍스트 데이터 (입력 문장)
sentences = [
    "자연어 처리는 재미있는 분야입니다.",
    "딥러닝은 많은 데이터를 필요로 합니다.",
    "한국어 NLP는 정말 재미있어요!"
]

In [3]:
# 2. 토크나이징 (Tokenizing)
okt = Okt()
tokenized_sentences = [okt.morphs(sentence) for sentence in sentences]
print("토크나이징 결과:", tokenized_sentences)

토크나이징 결과: [['자연어', '처리', '는', '재미있는', '분야', '입니다', '.'], ['딥', '러닝', '은', '많은', '데이터', '를', '필요', '로', '합니다', '.'], ['한국어', 'NLP', '는', '정말', '재미있어요', '!']]


In [4]:
# 3. 인코딩 (Encoding): 단어를 숫자로 변환
tokenizer = Tokenizer()
tokenizer.fit_on_texts(tokenized_sentences)
encoded_sentences = tokenizer.texts_to_sequences(tokenized_sentences)
print("인코딩 결과:", encoded_sentences)

인코딩 결과: [[3, 4, 1, 5, 6, 7, 2], [8, 9, 10, 11, 12, 13, 14, 15, 16, 2], [17, 18, 1, 19, 20, 21]]


In [5]:
# 4. 패딩 (Padding): 길이를 맞추기 위해 0으로 채우기
max_len = 10  # 최대 길이 설정
padded_sentences = pad_sequences(encoded_sentences, maxlen=max_len, padding='post')
print("패딩 결과:", padded_sentences)

패딩 결과: [[ 3  4  1  5  6  7  2  0  0  0]
 [ 8  9 10 11 12 13 14 15 16  2]
 [17 18  1 19 20 21  0  0  0  0]]


In [6]:
# 5. 임베딩 (Embedding)
vocab_size = len(tokenizer.word_index) + 1  # 단어 사전 크기
embedding_dim = 8  # 임베딩 차원 크기

In [7]:
# 간단한 임베딩 모델 생성
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
model.compile('rmsprop', 'mse')  # 옵티마이저(rmsprop), 손실함수(mse)

c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
# 패딩된 문장을 임베딩 층에 통과
embeddings = model.predict(padded_sentences)
print("임베딩 결과 (첫 번째 문장):\n", embeddings[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
임베딩 결과 (첫 번째 문장):
 [[ 0.01660526  0.03738794 -0.00438107 -0.02312642  0.01658035  0.04571761
   0.03607336  0.01296366]
 [ 0.02802673  0.03849525  0.01809334  0.01801581 -0.04971021  0.04408402
   0.02073273  0.02635889]
 [ 0.00042003  0.04777299  0.01102956 -0.04803744  0.03396566 -0.04182193
   0.0272701   0.00472737]
 [-0.02521837 -0.02039329 -0.03225292 -0.02231213 -0.02967321  0.02742374
   0.03450418  0.04699713]
 [ 0.04989832 -0.01452028  0.02720776 -0.00304966  0.03946494  0.02049777
  -0.01957082 -0.01320983]
 [-0.04600542 -0.04229983  0.00978404 -0.03066096 -0.01209778 -0.01930902
   0.04706247 -0.03229779]
 [ 0.00849406 -0.02356904  0.04296649  0.0329258  -0.04843106 -0.03099746
   0.01005946  0.02821672]
 [-0.04570898 -0.03564658 -0.01952463  0.00772778 -0.00811822  0.02379515
   0.04807686 -0.03544437]
 [-0.04570898 -0.03564658 -0.01952463  0.00772778 -0.00811822  0.02379515
   0.04807686 -0.03544437]
 [-0.04570898 -0.03564658 -0.0195

In [9]:
### transformers 라이브러리 사전 설치 : pip install transformers
### tf-keras 라이브러리 사전 설치 : pip install tf-keras
### torch 설치 : pip install torch

# BERT 모델 예시
from transformers import pipeline

print("BERT (fill-mask) PyTorch 파이프라인 로딩...")
unmasker = pipeline(
    'fill-mask',
    model='klue/bert-base'
)
print("BERT 모델 로딩 완료.")

# [MASK] 토큰이 있는 문장
text = f"대한민국의 수도는 {unmasker.tokenizer.mask_token}입니다."  # 마스크 토큰에 할당된 문자열 속성, 문장의 빈칸 [MASK]을 표시
print(f"\n입력: {text}")

# 2. 추론 수행
results = unmasker(text, top_k=3)  # 모델이 예측한 수많은 단어 후보 중 가장 높은 확률(점수)을 가진 상위 3개의 결과만 반환

# 3. 결과 출력
print("\n--- BERT 예측 결과 ---")
for res in results:
    print(f"  토큰: {res['token_str']:<5} (점수: {res['score']:.4f})")   # < : 왼쪽정렬, 5: 텍스트 총너비(5칸)

# 4. 최종 선택된 결과 출력
# results는 점수 순으로 정렬되어 있으므로 첫 번째 항목(results[0])이 최종 선택입니다.
best_result = results[0]

print("\n--- 최종 선택 (Best Pick) ---")
print(f"선택된 토큰: {best_result['token_str']}")
print(f"신뢰도 점수: {best_result['score']:.4f}")
print(f"완성된 문장: {best_result['sequence']}")

c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERT (fill-mask) PyTorch 파이프라인 로딩...


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 5901.55it/s]
BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT 모델 로딩 완료.

입력: 대한민국의 수도는 [MASK]입니다.

--- BERT 예측 결과 ---
  토큰: 서울    (점수: 0.5969)
  토큰: 광화문   (점수: 0.0635)
  토큰: 부산    (점수: 0.0467)

--- 최종 선택 (Best Pick) ---
선택된 토큰: 서울
신뢰도 점수: 0.5969
완성된 문장: 대한민국의 수도는 서울 입니다.


In [10]:
# BERT 모델 예시
# 감정분석(sentiment-analysis)
from transformers import pipeline

In [11]:
classifier = pipeline("sentiment-analysis")
classifier("오늘 기분이 좋아요")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4144.65it/s]


[{'label': 'POSITIVE', 'score': 0.8848781585693359}]

In [12]:
classifier = pipeline("sentiment-analysis")
classifier("오늘 기분이 좋아요")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6037.56it/s]


[{'label': 'POSITIVE', 'score': 0.8848781585693359}]

In [13]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

[{'label': 'POSITIVE', 'score': 0.9598049521446228},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [14]:
# BERT 모델 예시
# question-answering
question_answerer = pipeline("question-answering")
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [15]:
# GPT 모델 예시
# 텍스트 생성(text generation)
generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\human-31\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM3-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/

[{'generated_text': "In this course, we will teach you how to do several things with Python, including:\n- Create a new Python script\n- Use the Python interpreter\n- Create a variable\n- Use data types such as integers, floats, strings\n- Use operators such as +, -, *, /, //, % \n- Use if statements\n- Use for loops\n- Use while loops\n- Use functions\n- Use lists\n- Use dictionaries\n- Use sets\n- Use file I/O\n- Use the built-in functions and modules\n- Use error handling\n- Use object-oriented programming\n- Use regular expressions\n\nThis course is designed for beginners. You do not need to have any prior programming experience to take this course.\n\nWe will also include exercises for you to practice what you have learned.\n\nTo start, you will need to have Python installed on your computer.\n\nYou can download Python from the official Python website: https://www.python.org/downloads/\n\nOnce you have Python installed, open a new terminal or command prompt and type the following 

In [17]:
# BERT와 GPT 모델 특성을 필요로 한 예시
# 요약(Summarization)
summarizer = pipeline("summarization")  # 모델 미지정 시 sshleifer/distilbart-cnn-12-6 라는 모델을 로드
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [19]:
# 사전설치 : pip install faiss-cpu sentence_transformers langchain-community langchain-core
from langchain_community.vectorstores import FAISS   # 벡터 저장소
from langchain_core.output_parsers import StrOutputParser    # 문자열로 출력
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough    #입력데이터 전체를 다음단계로 전달
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
vectorstore = FAISS.from_texts(
    [
        "영준은 랭체인 주식회사에서 근무를 하였습니다.",
        "설현은 테디와 같은 회사에서 근무하였습니다.",
        "영준의 직업은 개발자입니다.",
        "설현의 직업은 디자이너입니다.",
    ],
    embedding = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask'),
)

C:\Users\human-31\AppData\Local\Temp\ipykernel_30932\3793202338.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask'),
c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\human-31\.cache\huggingface\hub\models--jhgan--ko-sroberta-multitask. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_S

In [1]:
# 사전 설치 : pip install langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu
# ollama 설치 : https://ollama.com/download/windows
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문맥이 가능한 한 유지되도록 청크 분할(문단, 줄바꿈, 공백 등)

c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 데이터 로드
loader = TextLoader("./dataset/history.txt", encoding='UTF8')  # 텍스트 파일 로드
documents = loader.load()  # 문서 로드

In [ ]:
# 2. 벡터 임베딩 생성 (Hugging Face 사용)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# vectorstore = FAISS.from_documents(documents, embeddings)
# 저장
vectorstore.save_local("faiss_index")